[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/gravitation.ipynb)

# Gravitation full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations**: the motion worked out step by step, a fraction of a second at a
time, and then checked against the formula the course derives. You can move three of them yourself: the **launch
speed** of a probe fired straight up, the **height** of an orbit, and the **farthest distance** of a planet's orbit.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. Big and small numbers in E notation: `6.42e23` for
6.42 × 10²³, `6.6743e-11` for 6.6743 × 10⁻¹¹ (`6.42*10^23` works too). You can type the arithmetic: `2.25^1.5`,
`sqrt(2*3.986e14/6.371e6)/1000`, `4*pi^2`. Numbers without thousands separators: `35793.5`, not `35,793.5`. For two
answers, separate them with a comma, in the order the question lists them (the box's label shows the order). You can
also put your answer between the quotes in a cell's last line, `practice("P1.4", "")`, and run the cell: it checks
straight away.

**The cheat sheet.** Every formula and constant from the course on two pages:
[gravitation cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/gravitation_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, FloatLogSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and paths in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# velocity SKY · acceleration MINT · force CORAL · energy VIOLET · bodies and paths CHALK · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the same as the course's: CODATA 2022 for G, IAU/IERS nominal values for Earth) ----
G = 6.6743e-11           # N m²/kg²
M = 5.972e24             # kg, Earth's mass
R = 6.371e6              # m, Earth's mean radius
g = 9.8                  # m/s², the value every question uses
GM = G * M               # 3.986 × 10¹⁴ m³/s²

# ---- the course's simulator: fourth-order Runge–Kutta, a fixed step at a time ----
def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """deriv(t, s) -> ds/dt as a tuple; state a tuple of numbers. Returns the list of (t, state) samples from 0 to
    t_end, or up to the first step where stop(t, s) is True, placed on the event by linear interpolation when stop
    names its test variable (stop.var = index, stop.level = the value it crosses)."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s2 = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t2 = t + dt
        if stop is not None and stop(t2, s2):
            i = getattr(stop, "var", None)
            if i is not None and s2[i] != s[i]:
                f = (getattr(stop, "level", 0.0) - s[i]) / (s2[i] - s[i])
                t2 = t + f * dt
                s2 = tuple(a + f * (b - a) for a, b in zip(s, s2))
            out.append((t2, s2))
            return out
        t, s = t2, s2
        out.append((t, s))
    return out

def close(a, b, rel=1e-4):
    """True when a simulated value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

def central(gm):
    """Motion in a plane round a body of G M = gm at the origin: state (x, y, vx, vy)."""
    def f(t, s):
        x, y, vx, vy = s
        r3 = (x * x + y * y) ** 1.5
        return (vx, vy, -gm * x / r3, -gm * y / r3)
    return f

def orbit_run(r0, v0, gm, t_end, dt):
    """Start at (r0, 0) moving (0, v0); stop when y crosses zero upwards after half the time: one full turn."""
    stop = lambda t, s: t > t_end / 2 and s[1] >= 0
    stop.var, stop.level = 1, 0.0
    return simulate(central(gm), (r0, 0.0, 0.0, v0), t_end, dt, stop)

def radial(t, s):
    """Straight up and down above Earth: state (r, v)."""
    return (s[1], -GM / s[0] ** 2)

# ---- the course's formulas ----
def force(m1, m2, r): return G * m1 * m2 / r ** 2                       # F = G m₁ m₂ / r²
def g_surface(Mp, Rp): return G * Mp / Rp ** 2                           # g = G M / R²
def g_height(h): return g * (R / (R + h)) ** 2                           # g R² / (R + h)²
def g_height_approx(h): return g * (1 - 2 * h / R)                       # g (1 − 2h/R), for h much smaller than R
def g_depth(d): return g * (1 - d / R)                                   # g (1 − d/R), a uniform ball
def v_escape(gm=GM, Rp=R): return math.sqrt(2 * gm / Rp)                 # √(2 G M / R)
def v_orbit(r, gm=GM): return math.sqrt(gm / r)                           # √(G M / r)
def period(r, gm=GM): return 2 * math.pi * math.sqrt(r ** 3 / gm)        # T = 2π √(r³ / G M)
def radius_for(T, gm=GM): return (gm * T ** 2 / (4 * math.pi ** 2)) ** (1 / 3)
def energy(m, r, gm=GM): return -gm * m / (2 * r)                        # E = −G M m / 2r
def apex(v0): return R / (1 - v0 ** 2 * R / (2 * GM))                    # straight up below escape speed: the top

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers (6.42e23 too), + − × ÷, powers, brackets, sqrt(…) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(km/s|m/s²|m/s\^2|m/s2|mm/s²|mm/s\^2|mm/s2|m³/s²|km|GJ|MJ|nN|N|AU|kg|m|s|years?|minutes?|mins?|"
        r"seconds?|hours?|degrees?|°|%)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": "Kepler's laws",
     "q": "A small planet's orbit has a semi-major axis of 2.25 AU. Using T² = a³ (years, AU), what is its period in years?",
     "parts": ['period (years)'],
     "answer": [3.375], "tol": [0.005], "function": '^',
     "hint": 'T² = a³, so T = a^(3/2): powers with ^',
     "working": 'T = a^(3/2) = 2.25^1.5 = 1.5³ = 3.375 years'},
    {"id": 'P1.2', "tier": 1, "group": "Kepler's laws",
     "q": 'A comet is 2 AU from the Sun at its closest and 6 AU at its farthest. How many times faster is it at its closest point?',
     "parts": ['how many times faster'],
     "answer": [3.0], "tol": [0.005], "function": '—',
     "hint": 'equal areas: r_p v_p = r_a v_a',
     "working": 'Equal areas: r_p v_p = r_a v_a, so v_p/v_a = r_a/r_p = 6/2 = 3'},
    {"id": 'P1.3', "tier": 1, "group": 'The universal law of gravitation',
     "q": 'Two 5 kg balls sit with their centres 0.5 m apart. What is the pull between them, in nanonewtons?',
     "parts": ['the pull (nN)'],
     "answer": [6.6743], "tol": [0.005], "function": '—',
     "hint": 'F = G m₁m₂ ÷ r²; 1 N = 10⁹ nN',
     "working": 'F = G m₁m₂/r² = 6.6743 × 10⁻¹¹ × 25/0.25 = 6.6743e-09 N = 6.6743 nN'},
    {"id": 'P1.4', "tier": 1, "group": 'The universal law of gravitation',
     "q": 'Two bodies pull each other with 90 N. The distance between them is tripled. What is the new pull, in newtons?',
     "parts": ['the new pull (N)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "hint": 'the pull goes as 1 ÷ r²',
     "working": 'Inverse square: 90/3² = 10 N'},
    {"id": 'P1.5', "tier": 1, "group": 'g, and how it changes',
     "q": 'A planet has a mass of 6.42 × 10²³ kg and a radius of 3,396 km. What is g on its surface, in metres per second squared?',
     "parts": ['g (m/s²)'],
     "answer": [3.71539839012432], "tol": [0.005], "function": '—',
     "hint": 'g = G M ÷ R², R in metres',
     "working": 'g = G M/R² = 6.6743 × 10⁻¹¹ × 6.42 × 10²³/(3.396 × 10⁶)² = 3.7154 m/s²'},
    {"id": 'P1.6', "tier": 1, "group": 'g, and how it changes',
     "q": "Taking g = 9.8 m/s² at the surface, what is g at a height equal to Earth's radius, in metres per second squared?",
     "parts": ['g at a height of R (m/s²)'],
     "answer": [2.45], "tol": [0.005], "function": '—',
     "hint": 'g R² ÷ (R + h)²',
     "working": 'g R²/(2R)² = g/4 = 9.8/4 = 2.45 m/s²'},
    {"id": 'P1.7', "tier": 1, "group": 'g, and how it changes',
     "q": 'Taking Earth as a uniform ball and g = 9.8 m/s², what is g a quarter of the way to the centre, in metres per second squared?',
     "parts": ['g a quarter of the way down (m/s²)'],
     "answer": [7.3500000000000005], "tol": [0.005], "function": '—',
     "hint": 'g (1 − d ÷ R)',
     "working": 'g(1 − d/R) = 9.8 × 3/4 = 7.35 m/s²'},
    {"id": 'P1.8', "tier": 1, "group": 'Gravitational potential energy',
     "q": "What is the gravitational potential energy of 1 kg on Earth's surface, in megajoules?",
     "parts": ['U (MJ)'],
     "answer": [-62.563050698477475], "tol": [0.005], "function": '—',
     "hint": 'U = −G M m ÷ R',
     "working": 'U = −G M m/R = −3.986 × 10¹⁴/6.371 × 10⁶ = -6.2563e+07 J = -62.5631 MJ'},
    {"id": 'P1.9', "tier": 1, "group": 'Escape speed',
     "q": "What is the escape speed from Earth's surface, in kilometres per second? (G M = 6.6743 × 10⁻¹¹ × 5.972 × 10²⁴ = 3.986 × 10¹⁴ m³/s², R = 6,371 km)",
     "parts": ['escape speed (km/s)'],
     "answer": [11.18597789184991], "tol": [0.005], "function": 'SQRT',
     "hint": 'v = √(2 G M ÷ R): SQRT',
     "working": 'v = √(2 G M/R) = √(2 × 3.986 × 10¹⁴/6.371 × 10⁶) = 11186.0 m/s = 11.1860 km/s'},
    {"id": 'P1.10', "tier": 1, "group": 'Earth satellites',
     "q": 'What is the speed of an orbit just above the ground (no air), in kilometres per second?',
     "parts": ['orbital speed (km/s)'],
     "answer": [7.909680821529872], "tol": [0.005], "function": 'SQRT',
     "hint": 'v = √(G M ÷ R): SQRT',
     "working": 'v = √(G M/R) = 7909.7 m/s = 7.9097 km/s (escape is √2 times this)'},
    {"id": 'P2.1', "tier": 2, "group": 'The universal law of gravitation',
     "q": 'The Moon is 384,400 km away and goes round in 27.3217 days. What is its acceleration towards Earth, in millimetres per second squared?',
     "parts": ['acceleration (mm/s²)'],
     "answer": [2.723329522271009], "tol": [0.005], "function": 'PI',
     "hint": 'a = 4π² r ÷ T², T in seconds: PI()',
     "working": 'a = 4π² r/T² = 4π² × 3.844 × 10⁸/(27.3217 × 86,400)² = 0.002723 m/s² = 2.7233 mm/s² (g/3,600 = 2.7222 mm/s²)'},
    {"id": 'P2.2', "tier": 2, "group": 'The universal law of gravitation',
     "q": 'What is the pull between Earth (5.972 × 10²⁴ kg) and the Moon (7.346 × 10²² kg) at 384,400 km, in units of 10²⁰ N?',
     "parts": ['the pull (in 10²⁰ N)'],
     "answer": [1.9815712324191872], "tol": [0.005], "function": '—',
     "hint": 'F = G M m ÷ r²',
     "working": 'F = G M m/r² = 1.9816e+20 N'},
    {"id": 'P2.3', "tier": 2, "group": 'The universal law of gravitation',
     "q": "Where between Earth and the Moon do their pulls on a spacecraft cancel? Give the distance from Earth's centre in thousands of kilometres.",
     "parts": ["distance from Earth's centre (thousand km)"],
     "answer": [346.0230391784831], "tol": [0.005], "function": 'SQRT',
     "hint": 'G M ÷ x² = G m ÷ (r − x)²: SQRT',
     "working": 'G M/x² = G m/(r − x)² → x/(r − x) = √(M/m) = 9.0164 → x = r/(1 + √(m/M)) = 346,023.0 km'},
    {"id": 'P2.4', "tier": 2, "group": 'The universal law of gravitation',
     "q": "Earth circles the Sun at 1.49598 × 10¹¹ m once every 365.256 days. Find the Sun's mass in units of 10³⁰ kg.",
     "parts": ["the Sun's mass (in 10³⁰ kg)"],
     "answer": [1.9884248079799676], "tol": [0.005], "function": 'PI',
     "hint": 'T² = 4π² r³ ÷ (G M), so M = 4π² r³ ÷ (G T²): PI()',
     "working": 'T² = 4π² r³/(G M) → M = 4π² r³/(G T²) = 1.9884e+30 kg'},
    {"id": 'P2.5', "tier": 2, "group": 'g, and how it changes',
     "q": 'Taking g = 9.8 m/s² at the surface, what is g 400 km up, in metres per second squared? Use the exact formula.',
     "parts": ['g at 400 km (m/s²)'],
     "answer": [8.676321937399678], "tol": [0.005], "function": '—',
     "hint": 'g R² ÷ (R + h)²',
     "working": 'g R²/(R + h)² = 9.8 × (6,371/6,771)² = 8.6763 m/s²'},
    {"id": 'P2.6', "tier": 2, "group": 'g, and how it changes',
     "q": 'At what depth (in kilometres) is g the same as 5 km above the ground? Use g(1 − 2h/R) and g(1 − d/R).',
     "parts": ['depth (km)'],
     "answer": [10.0], "tol": [0.005], "function": '—',
     "hint": 'g (1 − 2h ÷ R) = g (1 − d ÷ R)',
     "working": '1 − 2h/R = 1 − d/R → d = 2h = 10 km'},
    {"id": 'P2.7', "tier": 2, "group": 'Gravitational potential energy',
     "q": 'How much energy does lifting 500 kg from the ground to 1,000 km up take, in gigajoules? Use the exact form with g = 9.8 m/s².',
     "parts": ['energy (GJ)'],
     "answer": [4.235232668566002], "tol": [0.005], "function": '—',
     "hint": 'ΔU = m g h R ÷ (R + h)',
     "working": 'ΔU = m g h R/(R + h) = 500 × 9.8 × 10⁶ × 6,371/7,371 = 4.2352e+09 J (m g h alone: 4.900e+09 J)'},
    {"id": 'P2.8', "tier": 2, "group": 'Escape speed',
     "q": 'The Moon has a mass of 7.346 × 10²² kg and a radius of 1,737.4 km. What is its escape speed, in kilometres per second?',
     "parts": ['escape speed (km/s)'],
     "answer": [2.375709930670475], "tol": [0.005], "function": 'SQRT',
     "hint": 'v = √(2 G m ÷ R): SQRT',
     "working": '√(2 G m/R) = 2375.7 m/s = 2.3757 km/s'},
    {"id": 'P2.9', "tier": 2, "group": 'Earth satellites',
     "q": 'A satellite circles 800 km above the ground. What is its speed (km/s) and its period (minutes)?',
     "parts": ['speed (km/s)', 'period (minutes)'],
     "answer": [7.4554334880896365, 100.72457425349532], "tol": [0.005, 0.005], "function": 'SQRT, PI',
     "hint": 'r = R + h; v = √(G M ÷ r); T = 2π r ÷ v: SQRT, PI()',
     "working": 'r = 7,171 km; v = √(G M/r) = 7.4554 km/s; T = 2π r/v = 100.7246 min'},
    {"id": 'P2.10', "tier": 2, "group": 'Earth satellites',
     "q": 'What is the total energy of a 2,000 kg satellite circling 500 km up, in gigajoules?',
     "parts": ['total energy (GJ)'],
     "answer": [-58.01036181050794], "tol": [0.005], "function": '—',
     "hint": 'E = −G M m ÷ 2r, r = R + h',
     "working": 'E = −G M m/2r = −3.986 × 10¹⁴ × 2,000/(2 × 6.871 × 10⁶) = -58.0104 GJ'},
    {"id": 'P3.1', "tier": 3, "group": "Kepler's laws",
     "q": 'A distant body takes 125 years to go round the Sun. What is the semi-major axis of its orbit, in AU?',
     "parts": ['semi-major axis (AU)'],
     "answer": [24.999999999999996], "tol": [0.005], "function": '^',
     "hint": 'T² = a³, so a = T^(2/3)',
     "working": 'a = T^(2/3) = 125^(2/3) = 5² = 25 AU'},
    {"id": 'P3.2', "tier": 3, "group": 'g, and how it changes',
     "q": 'At what height (km) is g one ninth of its value on the ground?',
     "parts": ['height (km)'],
     "answer": [12742.0], "tol": [0.005], "function": '—',
     "hint": '(R ÷ (R + h))² = 1/9',
     "working": '(R/(R + h))² = 1/9 → R + h = 3R → h = 2R = 12,742 km'},
    {"id": 'P3.3', "tier": 3, "group": 'g, and how it changes',
     "q": 'At 1,000 km up, by what percent does g(1 − 2h/R) fall short of the exact value? (g = 9.8 m/s²)',
     "parts": ['how far short (%)'],
     "answer": [8.16445373624295], "tol": [0.005], "function": '—',
     "hint": 'exact g R² ÷ (R + h)² against g (1 − 2h ÷ R)',
     "working": 'exact 7.3213, approximate 6.7236 m/s²: short by 8.1645 %: h is not small next to R'},
    {"id": 'P3.4', "tier": 3, "group": 'Escape speed',
     "q": 'A probe is fired straight up at 8 km/s (no air). How high does it rise, in kilometres?',
     "parts": ['height (km)'],
     "answer": [6670.538291851739], "tol": [0.005], "function": '—',
     "hint": '½ v² − G M ÷ R = −G M ÷ r',
     "working": '½ v² − G M/R = −G M/r → r = R/(1 − v² R/2G M) = 13,041.54 km; height 6,670.5383 km'},
    {"id": 'P3.5', "tier": 3, "group": 'Escape speed',
     "q": 'Mars has a radius of 3,396 km and an escape speed of 5.0 km/s. Estimate its mass in units of 10²³ kg.',
     "parts": ["Mars's mass (in 10²³ kg)"],
     "answer": [6.360217550904216], "tol": [0.005], "function": '—',
     "hint": 'v² = 2 G M ÷ R',
     "working": 'v² = 2 G M/R → M = v² R/(2G) = 6.3602e+23 kg (measured 6.42 × 10²³: 5.0 is rounded)'},
    {"id": 'P3.6', "tier": 3, "group": 'Earth satellites',
     "q": 'A 1,000 kg satellite moves from 400 km up to 800 km up. How much energy must be added (GJ), and by how much does its speed change (km/s; negative if it slows)?',
     "parts": ['energy added (GJ)', 'speed change (km/s; minus if it slows)'],
     "answer": [1.6418103233002053, -0.2170569251939678], "tol": [0.005, 0.005], "function": '—',
     "hint": 'E = −G M m ÷ 2r at each height; v = √(G M ÷ r)',
     "working": 'ΔE = G M m/2 × (1/r₁ − 1/r₂) = 1.6418 GJ added; speed 7.6725 → 7.4554 km/s: it slows by 0.2171'},
    {"id": 'P3.7', "tier": 3, "group": 'Earth satellites',
     "q": 'What height (km) gives an orbit of exactly 2 hours?',
     "parts": ['height (km)'],
     "answer": [1687.921515775824], "tol": [0.005], "function": 'PI, ^',
     "hint": 'r = (G M T² ÷ 4π²)^(1/3); h = r − R: PI()',
     "working": 'r = (G M T²/4π²)^(1/3) = 8,058.92 km; h = 1,687.9215 km'},
    {"id": 'P3.8', "tier": 3, "group": 'Earth satellites',
     "q": 'A friend says a 2,000 kg satellite needs a higher speed than a 1,000 kg one to circle 600 km up. What speed does each need, in kilometres per second?',
     "parts": ['the speed each needs (km/s)'],
     "answer": [7.561626465718722], "tol": [0.005], "function": 'SQRT',
     "hint": "v = √(G M ÷ r): where is the satellite's mass?",
     "working": "v = √(G M/r): the satellite's own mass cancels, so both need 7.5616 km/s"},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by a comma",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ both right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

assert round(GM / 1e14, 3) == 3.986 and round(G * M / R ** 2, 2) == 9.82
print("Ready: G M =", f"{GM:.4e}", "m³/s²; g from G, M and R =", round(G * M / R ** 2, 2), "m/s²")

## Tier 1 · Warm-up

One idea, one step: Kepler's third law in years and astronomical units, equal areas, the pull between two balls, the
inverse square, g on another planet, g above and below the ground, the potential energy of a kilogram, escape speed,
and the speed of the lowest orbit.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `force(m1, m2, r)`, `g_height(h)`,
`v_escape()`, `v_orbit(r)` and the rest. Powers are `**` in Python (`2.25 ** 1.5`), and `math.sqrt` is the square
root. Work in metres, kilograms and seconds, then change the unit at the end (divide by 1,000 for kilometres). Each line
below computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.1  T = 2.25 ** 1.5 = {2.25 ** 1.5} years")
print(f"P1.3  F = {force(5, 5, 0.5):.4e} N = {force(5, 5, 0.5) * 1e9:.4f} nN")
print(f"P1.5  g = {g_surface(6.42e23, 3.396e6):.4f} m/s²")
print(f"P1.9  escape speed = {v_escape() / 1e3:.4f} km/s;  P1.10  lowest orbit = {v_orbit(R) / 1e3:.4f} km/s")
assert agrees("P1.1", [2.25 ** 1.5]) and agrees("P1.2", [6 / 2])
assert agrees("P1.3", [force(5, 5, 0.5) * 1e9]) and agrees("P1.4", [90 / 3 ** 2])
assert agrees("P1.5", [g_surface(6.42e23, 3.396e6)])
assert agrees("P1.6", [g_height(R)]) and agrees("P1.7", [g_depth(R / 4)])
assert agrees("P1.8", [-GM * 1 / R / 1e6])
assert agrees("P1.9", [v_escape() / 1e3]) and agrees("P1.10", [v_orbit(R) / 1e3])
assert round(v_escape() / v_orbit(R), 9) == round(math.sqrt(2), 9)       # escape speed is √2 times the lowest orbit's
print("Every tier 1 answer, worked out in Python, matches the practice set.")

### g against the distance from Earth's centre

Inside Earth (taken as a uniform ball), g grows in a straight line from zero at the centre: g × r ÷ R. Outside, it
falls as one over the distance squared: g × (R ÷ r)². At twice the radius it is a quarter, 2.45 m/s². The dashed line
is the shortcut g (1 − 2h ÷ R): close near the ground, then wrong.

In [ ]:
rr = np.linspace(0, 4, 401)                                       # distance from the centre, in Earth radii
gg = np.where(rr < 1, g * rr, g / np.maximum(rr, 1) ** 2)
fig, ax = plt.subplots(figsize=(11.5, 4.4))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.16)
ax.plot(rr, gg, color=MINT, label="g (exact)")
near = rr[(rr >= 1) & (rr <= 1.6)]
ax.plot(near, g * (1 - 2 * (near - 1)), color=DIM, ls="--", lw=1.8, label="shortcut g (1 − 2h/R)")
ax.axvline(1, color=GHOST, lw=1.4); ax.text(1.03, 9.9, "the ground", color=DIM)
ax.plot([2], [g / 4], "o", color=GOLD, ms=9); ax.annotate("2.45 m/s² at 2R", (2, g / 4), (2.15, 4.2), color=GOLD)
ax.set_xlim(0, 4); ax.set_ylim(0, 10.5); ax.set_xlabel("distance from Earth's centre, in Earth radii")
ax.set_ylabel("g (m/s²)"); ax.legend(frameon=False, loc="upper right")
fig.suptitle("g rises in a straight line inside, then falls as 1/r² outside", x=0.08, ha="left", fontsize=15, color=GOLD)
footnote(fig, "computed from the course's formulas"); plt.show()
assert g_height(R) == g / 4 and g_depth(R / 2) == g / 2

## Kepler's laws, simulated

An invented small planet goes round the Sun: 1 AU at its closest, 3 AU at its farthest. In astronomical units and
years, the Sun's G M is 4π², so the third law reads T² = a³. The simulator works the orbit out step by step. It checks
three things: one turn takes a^(3/2) = 2.83 years (a = 2 AU), the eight stretches of equal time sweep equal areas
(the first law's ellipse and the second law together), and the planet moves three times as fast at its closest
point as at its farthest. Slide the farthest distance and run it again.

In [ ]:
GM_SUN_AU = 4 * math.pi ** 2                                       # AU³/year²

def kepler_picture(farthest=3.0):
    rp, ra = 1.0, farthest
    a = (rp + ra) / 2
    vp = math.sqrt(GM_SUN_AU * (2 / rp - 1 / a))                   # the speed at the closest point (AU/year)
    T = a ** 1.5
    run = orbit_run(rp, vp, GM_SUN_AU, 1.25 * T, T / 6000)
    t = np.array([p[0] for p in run]); x = np.array([p[1][0] for p in run]); y = np.array([p[1][1] for p in run])
    swept = np.concatenate([[0], np.cumsum(0.5 * (x[:-1] * y[1:] - x[1:] * y[:-1]))])     # area swept so far
    slices = np.diff(np.interp(T * np.arange(9) / 8, t, swept))          # the area swept in each eighth of the year
    marks = [min(int(np.searchsorted(t, T * k / 8)), len(t) - 1) for k in range(9)]
    fig, ax = plt.subplots(figsize=(8.5, 6.4))
    fig.subplots_adjust(left=0.1, right=0.97, top=0.86, bottom=0.1)
    for k in range(8):
        i0, i1 = marks[k], min(marks[k + 1], len(t) - 1)
        px, py = np.r_[0, x[i0:i1 + 1], 0], np.r_[0, y[i0:i1 + 1], 0]
        ax.fill(px, py, color=GOLD if k == 0 else (GHOST if k % 2 else "#2c3a36"), alpha=0.9 if k == 0 else 1, lw=0)
    ax.plot(x, y, color=CHALK, lw=2)
    ax.plot([0], [0], "o", color=GOLD, ms=14); ax.text(0.12, 0.12, "Sun", color=GOLD)
    ax.set_aspect("equal"); ax.set_xlabel("AU"); ax.set_ylabel("AU")
    fig.suptitle(f"one turn: {t[-1]:.3f} years (a^1.5 = {T:.3f}); eight equal-time areas "
                 f"{min(slices):.3f}–{max(slices):.3f} AU²", x=0.1, ha="left", fontsize=13, color=GOLD)
    footnote(fig); plt.show()
    return t[-1], T, slices, math.pi * a * a * math.sqrt(1 - ((ra - rp) / (ra + rp)) ** 2)

T_sim, T_law, slices, area = kepler_picture(3.0)
assert close(T_sim, T_law) and round(T_law, 2) == 2.83
assert all(close(s, area / 8, 1e-3) for s in slices)                 # equal areas in equal times

In [ ]:
interact(kepler_picture, farthest=FloatSlider(value=3.0, min=1.0, max=6.0, step=0.25, description="farthest (AU)"));

### The third law on six planets

NASA's planetary fact sheet (retrieved 2026-11-11) gives each planet's mean distance from the Sun and its period. In
AU and years, T² ÷ a³ comes out as 1 for every one of them (Saturn 0.99, from the table's rounding).

In [ ]:
PLANETS = [("Mercury", 57.9, 88.0), ("Venus", 108.2, 224.7), ("Earth", 149.6, 365.2), ("Mars", 228.0, 687.0),
           ("Jupiter", 778.5, 4331.0), ("Saturn", 1432.0, 10747.0)]      # mean distance (million km), period (days)
print(f"{'planet':10s} {'a (AU)':>8s} {'T (years)':>10s} {'T²/a³':>7s}")
for name, d, T in PLANETS:
    a_, T_ = d / 149.6, T / 365.2
    print(f"{name:10s} {a_:8.3f} {T_:10.3f} {T_ ** 2 / a_ ** 3:7.3f}")
assert [round((T / 365.2) ** 2 / (d / 149.6) ** 3, 2) for _, d, T in PLANETS] == [1.0, 1.0, 1.0, 1.0, 1.0, 0.99]

## Tier 2 · Core

Several steps each: the Moon's acceleration from its orbit, the pull between Earth and the Moon, the point where their
pulls cancel, the Sun's mass from Earth's year, g 400 km up, a depth that matches a height, the energy of a big lift,
the Moon's escape speed, a satellite's speed and period, and its total energy.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Days into seconds first (× 86,400), kilometres into metres (× 1,000), then the formula. The Moon test is in here too:
the Moon's acceleration from its orbit, 4π² r ÷ T², against g ÷ 60², since the Moon is about 60 Earth radii away.
Both come out at 0.00272 m/s².

In [ ]:
r_moon, T_moon, m_moon, R_moon = 3.844e8, 27.3217 * 86400, 7.346e22, 1.7374e6
a_moon = 4 * math.pi ** 2 * r_moon / T_moon ** 2
print(f"the Moon's acceleration: {a_moon:.6f} m/s² from its orbit;  g / 60² = {g / 60 ** 2:.6f} m/s²")
assert round(a_moon * 1e4, 1) == round(g / 3600 * 1e4, 1) == 27.2
assert agrees("P2.1", [a_moon * 1e3])
assert agrees("P2.2", [force(M, m_moon, r_moon) / 1e20])
assert agrees("P2.3", [r_moon / (1 + math.sqrt(m_moon / M)) / 1e6])
a_earth, T_earth = 1.49598e11, 365.256 * 86400
M_sun = 4 * math.pi ** 2 * a_earth ** 3 / (G * T_earth ** 2)
print(f"the Sun's mass: {M_sun:.4e} kg, about {M_sun / M:,.0f} Earths")
assert agrees("P2.4", [M_sun / 1e30])
assert agrees("P2.5", [g_height(4e5)]) and agrees("P2.6", [2 * 5e3 / 1e3])
assert agrees("P2.7", [500 * g * 1e6 * R / (R + 1e6) / 1e9])
assert agrees("P2.8", [v_escape(G * m_moon, R_moon) / 1e3])
assert agrees("P2.9", [v_orbit(R + 8e5) / 1e3, period(R + 8e5) / 60])
assert agrees("P2.10", [energy(2000, R + 5e5) / 1e9])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

### Change the launch speed: the probe rises and falls

A probe is fired straight up from the ground, with no air. The simulator follows it a few seconds at a time, with
the pull G M ÷ r² getting weaker as it climbs. At 10 km/s it rises about 25,400 km and falls back. The formula from
the energy, r = R ÷ (1 − v²R ÷ 2GM), gives the same top. Slide the speed: above 11.19 km/s, the escape speed, it
never comes back.

In [ ]:
def launch_picture(speed=10.0):
    v0 = speed * 1000
    returns = v0 < v_escape()
    if returns:
        stop = lambda t, s: s[1] < 0 and s[0] < R                   # back on the ground
        stop.var, stop.level = 0, R
    else:
        stop = lambda t, s: s[0] > 10 * R                           # ten Earth radii out, still moving
        stop.var, stop.level = 0, 10 * R
    run = simulate(radial, (R, v0), 86400, 4.0, stop)
    t = np.array([p[0] for p in run]) / 3600
    r = np.array([p[1][0] for p in run]); v = np.array([p[1][1] for p in run])
    top = r.max()
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.5, 1]})
    fig.subplots_adjust(left=0.08, right=0.98, top=0.82, bottom=0.19, wspace=0.28)
    ax.plot(t, (r - R) / 1e3, color=CHALK, label="height, simulated")
    if returns:
        ax.axhline((apex(v0) - R) / 1e3, color=DIM, ls="--", lw=1.6, label="top, from the formula")
        i = int(np.argmax(r)); ax.plot([t[i]], [(top - R) / 1e3], "o", color=GOLD, ms=9)
        title = (f"{speed:.1f} km/s: up {(top - R) / 1e3:,.0f} km (formula {(apex(v0) - R) / 1e3:,.0f} km), "
                 f"back after {t[-1]:.2f} h")
    elif t[-1] < 24:
        title = f"{speed:.1f} km/s: escapes; still {v[-1] / 1e3:.2f} km/s at 10 Earth radii"
    else:
        title = f"{speed:.1f} km/s: still climbing after a day"
    ax.set_xlabel("time (hours)"); ax.set_ylabel("height (km)"); ax.legend(frameon=False, loc="lower center")
    ax2.plot(t, v / 1e3, color=SKY); ax2.axhline(0, color=DIM, lw=1)
    ax2.set_xlabel("time (hours)"); ax2.set_ylabel("speed upwards (km/s)")
    fig.suptitle(title, x=0.08, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()
    return run

run10 = launch_picture(10.0)
top10 = max(p[1][0] for p in run10)
print(f"at 10 km/s: top {top10 / 1e3:,.1f} km from the centre, formula {apex(1e4) / 1e3:,.1f} km")
assert close(top10, apex(1e4), 1e-5) and round((apex(1e4) - R) / 1e3, -1) == 25360

In [ ]:
interact(launch_picture, speed=FloatSlider(value=10.0, min=2.0, max=12.0, step=0.1, description="km/s"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, spot the trap: a semi-major axis from a period, the height where g is a
ninth, how far the shortcut falls short at 1,000 km, how high a probe at 8 km/s rises, the mass of Mars from its
escape speed, the energy and the speed when an orbit is raised, the height for a two-hour orbit, and whether a
heavier satellite needs a faster orbit.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is algebra first, then one line: a = T^(2/3), r = (G M T² ÷ 4π²)^(1/3). And P3.4 can be
checked a second way: simulate the probe at 8 km/s and read off its highest point.

In [ ]:
assert agrees("P3.1", [125 ** (2 / 3)]) and agrees("P3.2", [R * (math.sqrt(9) - 1) / 1e3])
ex_, ap_ = g_height(1e6), g_height_approx(1e6)
print(f"P3.3  at 1,000 km: exact {ex_:.4f}, shortcut {ap_:.4f} m/s², short by {100 * (ex_ - ap_) / ex_:.2f} %")
assert agrees("P3.3", [100 * (ex_ - ap_) / ex_])
stop = lambda t, s: s[1] < 0; stop.var, stop.level = 1, 0.0
top8 = simulate(radial, (R, 8.0e3), 86400, 1.0, stop)[-1][1][0]           # straight up at 8 km/s, to the top
print(f"P3.4  8 km/s: simulated top {(top8 - R) / 1e3:,.2f} km up; formula {(apex(8e3) - R) / 1e3:,.2f} km")
assert close(top8, apex(8e3), 1e-6) and agrees("P3.4", [(apex(8e3) - R) / 1e3])
assert agrees("P3.5", [5.0e3 ** 2 * 3.396e6 / (2 * G) / 1e23])
assert agrees("P3.6", [(energy(1000, R + 8e5) - energy(1000, R + 4e5)) / 1e9, (v_orbit(R + 8e5) - v_orbit(R + 4e5)) / 1e3])
assert agrees("P3.7", [(radius_for(2 * 3600) - R) / 1e3])
assert agrees("P3.8", [v_orbit(R + 6e5) / 1e3])                          # the satellite's own mass is nowhere in it
print("Every tier 3 answer, worked out in Python, matches the practice set.")

### Change the height: watch the period change

A satellite in a circular orbit, simulated for one full turn. On the left, the orbit to scale round Earth, with its
velocity (sky) and the pull towards Earth's centre (coral). On the right, the period against the height from the
formula T = 2π √(r³ ÷ G M), and the simulated turn on it in gold. At 400 km one turn takes 92.4 minutes; at about
35,800 km it takes one sidereal day, the weather camera's orbit. Slide the height.

In [ ]:
def orbit_picture(height=400.0):
    r0 = R + height * 1e3
    T = period(r0)
    run = orbit_run(r0, v_orbit(r0), GM, 1.25 * T, T / 3000)
    x = np.array([p[1][0] for p in run]) / 1e3; y = np.array([p[1][1] for p in run]) / 1e3
    T_sim = run[-1][0]
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.5, 5.2), gridspec_kw={"width_ratios": [1, 1.2]})
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.17, wspace=0.3)
    ax.add_patch(plt.Circle((0, 0), R / 1e3, color=GHOST))
    ax.plot(x, y, color=CHALK, lw=2)
    span = r0 / 1e3
    ax.arrow(span, 0, 0, 0.35 * span, color=SKY, width=0.012 * span, length_includes_head=True)
    ax.arrow(span, 0, -0.3 * span, 0, color=CORAL, width=0.012 * span, length_includes_head=True)
    ax.set_xlim(-1.25 * span, 1.25 * span); ax.set_ylim(-1.25 * span, 1.25 * span); ax.set_aspect("equal")
    ax.set_xlabel("km"); ax.grid(False)
    hh = np.geomspace(200, 40000, 200)
    ax2.plot(hh, [period(R + h * 1e3) / 3600 for h in hh], color=DIM)
    ax2.plot([height], [T_sim / 3600], "o", color=GOLD, ms=10)
    ax2.set_xscale("log"); ax2.set_xlabel("height above the ground (km)"); ax2.set_ylabel("period (hours)")
    fig.suptitle(f"{height:,.0f} km up: {v_orbit(r0) / 1e3:.2f} km/s, one turn in {T_sim / 60:,.1f} min "
                 f"(formula {T / 60:,.1f} min)", x=0.07, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()
    return T_sim, T

T_sim, T_400 = orbit_picture(400.0)
assert close(T_sim, T_400) and round(T_400 / 60, 1) == 92.4 and round(v_orbit(R + 4e5) / 1e3, 2) == 7.67

In [ ]:
interact(orbit_picture, height=FloatLogSlider(value=400, base=10, min=math.log10(200), max=math.log10(40000),
                                              step=0.01, description="height (km)"));

### The weather camera: one sidereal day, simulated

The course's opening question: a weather camera must watch the same patch of ocean all day and all night. It must go
round once each time Earth turns once relative to the stars, in 23 h 56 min 4.09 s (86,164.09 s, the sidereal day),
above the equator. The period formula run backwards gives its orbit; the simulation follows it for one sidereal day.
Earth turns once in the same time, so the spot of ocean under the camera (gold) turns with it: at every moment the
camera, the spot and Earth's centre sit on one line.

In [ ]:
T_SID = 86164.09                                                      # s: one turn of Earth relative to the stars
r_geo = radius_for(T_SID); h_geo = r_geo - R; v_geo = v_orbit(r_geo)
print(f"r = {r_geo / 1e3:,.0f} km from the centre, h = {h_geo / 1e3:,.0f} km up, moving at {v_geo / 1e3:.2f} km/s")
assert round(r_geo / 1e3) == 42164 and round(h_geo / 1e3, -2) == 35800 and round(v_geo / 1e3, 2) == 3.07
geo = orbit_run(r_geo, v_geo, GM, 1.25 * T_SID, 10.0)
print(f"one turn of the camera, simulated: {geo[-1][0]:,.2f} s; the sidereal day: {T_SID:,.2f} s")
assert close(geo[-1][0], T_SID, 1e-6)

fig, ax = plt.subplots(figsize=(7.5, 7))
fig.subplots_adjust(left=0.1, right=0.97, top=0.88, bottom=0.12)
ax.add_patch(plt.Circle((0, 0), R / 1e3, color=GHOST))
gx = np.array([p[1][0] for p in geo]) / 1e3; gy = np.array([p[1][1] for p in geo]) / 1e3
ax.plot(gx, gy, color=CHALK, lw=1.8)
for k in range(8):
    i = min(int(round(k / 8 * (len(geo) - 1))), len(geo) - 1)
    t_, (x_, y_, _, _) = geo[i]
    turn = 2 * math.pi * t_ / T_SID                               # how far Earth has turned
    sx, sy = R / 1e3 * math.cos(turn), R / 1e3 * math.sin(turn)
    ax.plot([sx, x_ / 1e3], [sy, y_ / 1e3], color=GOLD, lw=1, ls="--")
    ax.plot([x_ / 1e3], [y_ / 1e3], "o", color=CHALK, ms=7); ax.plot([sx], [sy], "o", color=GOLD, ms=7)
ax.set_aspect("equal"); ax.set_xlabel("km"); ax.grid(False)
fig.suptitle("Eight moments of one sidereal day: the camera stays over its spot", x=0.1, ha="left", fontsize=14, color=GOLD)
footnote(fig); plt.show()

## Sources
Every exercise and set-up here is our own. G is the CODATA 2022
value (physics.nist.gov/cuu/Constants); Earth's mass and mean radius are the IAU/IERS nominal values, rounded. The
Moon's, Earth's and the planets' numbers are from NASA's NSSDC fact sheets (nssdc.gsfc.nasa.gov/planetary/factsheet,
retrieved 2026-11-11); the sidereal day, 86,164.09 s, agrees with NASA's 23.9345 hours. The simulator is the course's: fourth-order Runge–Kutta with a fixed step.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph07_gravitation/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph07_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"]) == (_e["q"], _e["working"], _e["function"], _e["tier"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (G, M, R, g) == (_P.G, _P.M, _P.R, _P.g) and _P.PLANETS == PLANETS
    assert (r_moon, 27.3217, m_moon, R_moon) == (_P.MOON["r"], _P.MOON["T_days"], _P.MOON["m"], _P.MOON["R"])
    assert T_SID == _P.SIDEREAL_DAY and close(r_geo, _P.GEO["r"], 1e-12) and close(geo[-1][0], _P.SIM["geo_T"], 1e-9)
    assert close(T_400, _P.SIM["T_400"]) and close(top10, _P.SIM["apex_r"], 1e-6)
    print("matches the course's practice set")